# I-S10 — ISLES 2022 SegResNet + Attention Gate

- Dataset: ISLES 2022
- Input: DWI + ADC (2 channels)
- Split: Train 175 / Validation 38 / Test 37
- Patch: 80 × 96 × 80
- DataLoader batch size: 1
- Patches per volume: 2
- Max epoch: 300
- Early stopping: patience 30 / min_delta 0.001
- 변경점: **I-S00 SegResNet baseline 대비 Additive Attention Gate만 추가**
- Canonical config: `configs/experiments/ischemia/I-S10.yaml`
- Canonical model implementation: `src/brainon_ai/models/ischemia/segresnet/network.py`
- Canonical training entrypoint: `src/brainon_ai/models/ischemia/segresnet/train.py`
- Canonical evaluation entrypoint: `src/brainon_ai/models/ischemia/segresnet/evaluate.py`

> **중요 — provenance**
>
> 실제 I-S10 학습은 기존 Colab notebook에서 수행되었습니다. 당시 notebook 내부에서 별도 `train_attention.py`를 생성하여 실행했습니다.
> 이후 동일한 Attention 구현과 실험 조건을 repository의 공통 `network.py`, `train.py`, `evaluate.py`로 승격했습니다.
> 아래 공통 파이프라인 실행 셀은 **재현용이며 이 정리본을 만들면서 다시 실행하지 않았습니다.**
>
> 실제 과거 학습 결과는 이 notebook 하단의 **Historical I-S10 run results**에 별도로 기록합니다.


## 1. Colab / repository / data 준비


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from google.colab import userdata
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

repo_dir = Path("/content/model")

# fresh runtime 기준. 이전 불완전 clone이 있으면 제거.
if repo_dir.exists() and not (repo_dir / ".git").exists():
    shutil.rmtree(repo_dir)

if not repo_dir.exists():
    token = userdata.get("git_token")
    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_TERMINAL_PROMPT"] = "0"

    askpass = "/tmp/git_askpass.sh"
    with open(askpass, "w") as f:
        f.write("""#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")
    os.chmod(askpass, 0o700)
    env["GIT_ASKPASS"] = askpass

    subprocess.run(
        [
            "git", "clone", "--branch", "dev",
            "https://github.com/brainCDSS/model.git",
            str(repo_dir),
        ],
        env=env,
        check=True,
    )

    os.remove(askpass)
    del token
    print("clone 완료")
else:
    print("기존 /content/model 저장소 사용")

print(
    subprocess.check_output(
        ["git", "-C", str(repo_dir), "branch", "--show-current"],
        text=True,
    ).strip()
)
print(
    subprocess.check_output(
        ["git", "-C", str(repo_dir), "log", "-1", "--oneline"],
        text=True,
    ).strip()
)
print("※ 이 Notebook에는 git add / commit / push 명령이 없습니다.")


In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Google Cloud 인증 완료")


In [ ]:
%%bash
set -e

mkdir -p /content/segresnet_data

gcloud storage cp --recursive   gs://brainon-cdss-datasets/ischemia/v1/SegResNet/train   gs://brainon-cdss-datasets/ischemia/v1/SegResNet/val   gs://brainon-cdss-datasets/ischemia/v1/SegResNet/test   gs://brainon-cdss-datasets/ischemia/v1/SegResNet/metadata   /content/segresnet_data/

echo "SegResNet 전처리 데이터 복사 완료"


In [ ]:
from pathlib import Path

root = Path("/content/segresnet_data")
expected = {"train": 175, "val": 38, "test": 37}

for split in ["train", "val", "test"]:
    adc = list((root / split / "images").glob("*_adc.nii.gz"))
    dwi = list((root / split / "images").glob("*_dwi.nii.gz"))
    mask = list((root / split / "labels").glob("*_mask.nii.gz"))

    print(
        f"{split.upper():5s} | "
        f"ADC: {len(adc):3d} | "
        f"DWI: {len(dwi):3d} | "
        f"MASK: {len(mask):3d}"
    )

    assert len(adc) == expected[split]
    assert len(dwi) == expected[split]
    assert len(mask) == expected[split]

print("\nDataset count: PASS")


In [ ]:
from pathlib import Path
import pandas as pd

data_root = Path("/content/segresnet_data")
rows = []

for split_folder, split_name in [
    ("train", "train"),
    ("val", "validation"),
    ("test", "test"),
]:
    image_dir = data_root / split_folder / "images"
    label_dir = data_root / split_folder / "labels"

    for dwi_path in sorted(image_dir.glob("*_dwi.nii.gz")):
        case_id = dwi_path.name.replace("_dwi.nii.gz", "")
        adc_path = image_dir / f"{case_id}_adc.nii.gz"
        mask_path = label_dir / f"{case_id}_mask.nii.gz"

        if not adc_path.is_file() or not mask_path.is_file():
            raise FileNotFoundError(case_id)

        rows.append({
            "case_id": case_id,
            "split": split_name,
            "dwi_path": str(dwi_path),
            "adc_path": str(adc_path),
            "mask_path": str(mask_path),
        })

manifest = pd.DataFrame(rows)
runtime_manifest = Path(
    "/content/segresnet_data/metadata/I-S10_runtime_manifest.csv"
)
manifest.to_csv(runtime_manifest, index=False)

nn_split = pd.read_csv(
    "/content/model/registry/splits/I-N00_v1.0.0.csv"
)[["case_id", "split"]].sort_values("case_id").reset_index(drop=True)

seg_split = manifest[["case_id", "split"]].sort_values(
    "case_id"
).reset_index(drop=True)

print("SegResNet I-S10 split:")
print(seg_split["split"].value_counts())
print()
print("Case별 Split 동일:", seg_split.equals(nn_split))

assert seg_split.equals(nn_split)
print("Split check: PASS")


In [ ]:
import torch

print("CUDA 사용 가능:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU를 사용할 수 없습니다.")

print("GPU :", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        1,
    ),
    "GB",
)


In [ ]:
!pip install -q monai nibabel pyyaml pandas


In [ ]:
import torch
import monai

print("PyTorch:", torch.__version__)
print("MONAI  :", monai.__version__)
print("GPU    :", torch.cuda.get_device_name(0))


## 2. Canonical repository source 확인


In [ ]:
from pathlib import Path

REPO_ROOT = Path("/content/model")
CONFIG_PATH = REPO_ROOT / "configs/experiments/ischemia/I-S10.yaml"
SEGRESNET_DIR = REPO_ROOT / "src/brainon_ai/models/ischemia/segresnet"
TRAIN_PY = SEGRESNET_DIR / "train.py"
EVALUATE_PY = SEGRESNET_DIR / "evaluate.py"
NETWORK_PY = SEGRESNET_DIR / "network.py"

DATA_ROOT = Path("/content/segresnet_data")
BACKUP_ROOT = Path(
    "/content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S10"
)
BEST_CKPT = BACKUP_ROOT / "checkpoints" / "checkpoint_best.pth"

for path in [CONFIG_PATH, TRAIN_PY, EVALUATE_PY, NETWORK_PY]:
    assert path.is_file(), path

print("Config   :", CONFIG_PATH)
print("Network  :", NETWORK_PY)
print("Train    :", TRAIN_PY)
print("Evaluate :", EVALUATE_PY)
print("Data root:", DATA_ROOT)
print("Run root :", BACKUP_ROOT)


### I-S00 ↔ I-S10 비교 원칙

I-S10은 I-S00과 다음 조건을 동일하게 유지하고 모델 구조의 Attention Gate만 변경합니다.

- DWI + ADC 입력
- Train / Validation / Test split
- preprocessing / patch / sampling
- loss / optimizer / scheduler
- epoch / early stopping
- full-volume sliding-window validation

현재 repository에서는 `I-S10.yaml`과 `network.py`가 모델 variant를 지정하고,
`train.py` / `evaluate.py` / 공통 data·loss·inference·metrics 모듈을 재사용합니다.


In [ ]:
import yaml

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    i_s10 = yaml.safe_load(f)

i_s00_path = REPO_ROOT / "configs/experiments/ischemia/I-S00.yaml"
with i_s00_path.open("r", encoding="utf-8") as f:
    i_s00 = yaml.safe_load(f)

print("I-S00 experiment_id:", i_s00["experiment_id"])
print("I-S00 model.name   :", i_s00["model"]["name"])
print("I-S10 experiment_id:", i_s10["experiment_id"])
print("I-S10 model.name   :", i_s10["model"]["name"])

# 실제 고정 조건 검증은 repository의 config.py가 담당합니다.


## 3. Sanity check — 공통 `train.py` 사용


In [ ]:
!python /content/model/src/brainon_ai/models/ischemia/segresnet/train.py \
  --config /content/model/configs/experiments/ischemia/I-S10.yaml \
  --data-root /content/segresnet_data \
  --backup-root /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S10 \
  --sanity-only


## 4. Training — 공통 `train.py` 사용


In [ ]:
!python /content/model/src/brainon_ai/models/ischemia/segresnet/train.py \
  --config /content/model/configs/experiments/ischemia/I-S10.yaml \
  --data-root /content/segresnet_data \
  --backup-root /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S10


## 5. Historical I-S10 run results

아래 값은 **실제 기존 Colab I-S10 학습 실행에서 나온 과거 결과**입니다.
현재 공통 `train.py` 셀을 실행해서 새로 생성한 결과가 아닙니다.

- Historical best checkpoint epoch: **140**
- Historical stored Validation Dice (38-case checkpoint selection metric): **0.7765787188944063**
- 당시 checkpoint 경로:
  `/content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S10/checkpoints/checkpoint_best.pth`

현재 연결된 Drive에서 해당 I-S10 checkpoint 실파일은 확인되지 않았으므로,
이 notebook 정리 과정에서는 checkpoint load / inference / 재평가를 실행하지 않았습니다.


## 6. Final validation — 공통 `evaluate.py` 사용

최종 reporting 정책:

- Validation 전체: **38 case**
- GT-positive: **37 case**
  - Dice / IoU / Precision / Recall / HD95 / ASSD
  - 최종 segmentation Macro metric은 37 case 평균
- GT-empty: **1 case**
  - segmentation Macro metric에서 제외
  - lesion absence / presence 판정으로 기록

아래 셀은 repository의 공통 evaluation entrypoint를 사용하는 재현용 명령입니다.
`checkpoint_best.pth`가 확보되어 있을 때만 실행합니다.


In [ ]:
!python /content/model/src/brainon_ai/models/ischemia/segresnet/evaluate.py \
  --data-root /content/segresnet_data \
  --checkpoint /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S10/checkpoints/checkpoint_best.pth \
  --split val \
  --output-dir /content/drive/MyDrive/BrainCDSS/ischemia/segresnet/I-S10/evaluation/validation/common_pipeline_reproduction


## 7. Historical final validation results

다음 값은 실제 기존 I-S10 Colab run에서 `checkpoint_best.pth`를 불러와 수행한
**Historical validation 결과**입니다. 이 정리본에서 재계산하지 않았습니다.

| Metric | GT-positive 37 case Macro |
|---|---:|
| Dice | **0.770540** |
| IoU | **0.646134** |
| Precision | **0.849337** |
| Recall | **0.753171** |
| HD95 | **9.188127 mm** |
| ASSD | **2.492853 mm** |

- GT-positive: **37**
- GT-empty: **1**
- Surface metric undefined: **0 / 37**
- Empty GT case: `sub-strokecase0170`
- Prediction: lesion absent (`pred_voxels = 0`)
- Result: **CORRECT**

Historical output paths recorded by the original run:

- `evaluation/validation/validation_positive37_case_metrics.csv`
- `evaluation/validation/validation_empty1_detection.csv`
- `evaluation/validation/validation_summary_positive37.json`


## 8. Historical lesion-size analysis

Size definition:

- Small: `< 5 mL`
- Medium: `5–20 mL`
- Large: `>= 20 mL`
- GT-empty case 제외

| Size group | n | Dice | IoU | Precision | Recall | HD95 (mm) | ASSD (mm) |
|---|---:|---:|---:|---:|---:|---:|---:|
| Small (<5 mL) | 16 | 0.719649 | 0.587612 | 0.843063 | 0.689856 | 15.641316 | 3.836441 |
| Medium (5–20 mL) | 15 | 0.802749 | 0.679727 | 0.834607 | 0.804509 | 3.736336 | 1.335905 |
| Large (>=20 mL) | 6 | 0.825727 | 0.718210 | 0.902889 | 0.793664 | 5.609102 | 1.802321 |

Historical run에서는 Small lesion에서 상대적으로 낮은 Recall과 높은 HD95가 관찰되었습니다.


## 9. Handoff

향후 I-S20 / I-S30 등 새로운 **모델 구조 variant**를 추가할 경우 기본 원칙은 다음과 같습니다.

1. 새 `I-Sxx.yaml` 추가
2. `network.py`에 모델 class / `build_model()` variant 등록
3. 기존 공통 `train.py`, `evaluate.py`, `data.py`, `losses.py`, `inference.py`, `metrics.py` 재사용

모델 구조 외의 실험 조건을 변경하는 실험은 해당 변경 항목을 명시적으로 config/source에 반영해야 합니다.
